# 05 Plot Results (Revised for Paper-Style Visualization)

This notebook keeps the original project paths and rewrites the plotting logic so the figures are easier to use in a report.

Core design choices:
- **Main figures**
  - final metric **dot plots** instead of dense grouped bars
  - test **PSNR convergence** arranged as **scene × model** panels, with colors for initialization setting
- **Appendix figures**
  - test **L1 convergence**
  - train/test overlay diagnostics
  - aggregate metric heatmaps
- **Tables**
  - tidy long-format summary
  - wide paper-style table
  - aggregate mean table

Suggested report usage:
- **Main text**: wide quantitative table, final metric dot plot, test PSNR convergence, qualitative rendering grid
- **Appendix**: L1 convergence, train/test overlay, aggregate heatmaps, extra qualitative cases

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CV_ROOT = Path('~/CV_Project').expanduser()
RESULTS_ROOT = CV_ROOT / 'results' / 'part1'
PLOTS_ROOT = CV_ROOT / 'plots' / 'part1'
MAIN_DIR = PLOTS_ROOT / 'main'
APP_DIR = PLOTS_ROOT / 'appendix'
TABLE_DIR = PLOTS_ROOT / 'tables'

for d in [MAIN_DIR, APP_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FINAL_CSV = RESULTS_ROOT / 'final' / 'final_metrics_18.csv'
CONV_CSV = RESULTS_ROOT / 'convergence' / 'all_convergence_metrics.csv'

final_df = pd.read_csv(FINAL_CSV)
conv_df = pd.read_csv(CONV_CSV)

print('final_df shape =', final_df.shape)
print('conv_df shape =', conv_df.shape)
print('FINAL_CSV =', FINAL_CSV)
print('CONV_CSV  =', CONV_CSV)

final_df shape = (18, 11)
conv_df shape = (720, 11)
FINAL_CSV = /home/bzhang512/CV_Project/results/part1/final/final_metrics_18.csv
CONV_CSV  = /home/bzhang512/CV_Project/results/part1/convergence/all_convergence_metrics.csv


## 1. Orders, labels, styling, and helper utilities

In [2]:
SCENE_ORDER = ['Re10k-1', 'DL3DV-2', '405841']
SETTING_ORDER = ['planA_colmap_full', 'planA_colmap_108', 'planB_vggt_108']
MODEL_ORDER = ['3dgs', 'scaffold']

METRICS = [
    ('ours_40000.PSNR', 'PSNR ↑', True),
    ('ours_40000.SSIM', 'SSIM ↑', True),
    ('ours_40000.LPIPS', 'LPIPS ↓', False),
]

SETTING_LABELS = {
    'planA_colmap_full': 'COLMAP full',
    'planA_colmap_108': 'COLMAP-108',
    'planB_vggt_108': 'VGGT-108',
}
MODEL_LABELS = {
    '3dgs': '3DGS',
    'scaffold': 'Scaffold-GS',
}
SCENE_LABELS = {
    'Re10k-1': 'Re10k-1',
    'DL3DV-2': 'DL3DV-2',
    '405841': '405841',
}

SETTING_COLORS = {
    'planA_colmap_full': '#1f4e79',
    'planA_colmap_108': '#2e8b57',
    'planB_vggt_108': '#8b1e3f',
}
MODEL_MARKERS = {
    '3dgs': 'o',
    'scaffold': 's',
}
MODEL_LINESTYLES = {
    '3dgs': '-',
    'scaffold': '--',
}

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'semibold',
})

def add_setting_key(df):
    df = df.copy()
    df['setting'] = df['plan'] + '_' + df['variant']
    return df

def apply_orders(df):
    df = df.copy()
    df['scene'] = pd.Categorical(df['scene'], categories=SCENE_ORDER, ordered=True)
    df['setting'] = pd.Categorical(df['setting'], categories=SETTING_ORDER, ordered=True)
    df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
    return df

def metric_best_mask(df, metric, higher_is_better=True):
    df = df.copy()
    best_vals = (
        df.groupby('scene')[metric].max()
        if higher_is_better
        else df.groupby('scene')[metric].min()
    )
    df['_is_best'] = df.apply(lambda r: np.isclose(r[metric], best_vals[r['scene']]), axis=1)
    return df['_is_best']

def flatten_columns(df):
    df = df.copy()
    df.columns = ['__'.join([str(x) for x in col if str(x) != '']) if isinstance(col, tuple) else str(col) for col in df.columns]
    return df

final_df = apply_orders(add_setting_key(final_df))
conv_df = apply_orders(add_setting_key(conv_df))

required_final_cols = {'scene', 'setting', 'model', 'run_name', 'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS'}
required_conv_cols = {'scene', 'setting', 'model', 'split', 'iter', 'psnr', 'l1'}

missing_final = required_final_cols - set(final_df.columns)
missing_conv = required_conv_cols - set(conv_df.columns)

assert not missing_final, f'Missing final_df columns: {missing_final}'
assert not missing_conv, f'Missing conv_df columns: {missing_conv}'

## 2. Quick sanity checks

In [3]:
display(
    final_df[['scene', 'setting', 'model', 'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']]
    .sort_values(['scene', 'setting', 'model'])
    .reset_index(drop=True)
)

display(
    conv_df.groupby(['scene', 'setting', 'model', 'split'])['iter']
    .agg(['min', 'max', 'count'])
    .reset_index()
    .sort_values(['scene', 'setting', 'model', 'split'])
)

,scene,setting,model,ours_40000.PSNR,ours_40000.SSIM,ours_40000.LPIPS
0,Re10k-1,planA_colmap_full,3dgs,35.164932,0.977586,0.042414
1,Re10k-1,planA_colmap_full,scaffold,34.862053,0.976933,0.043847
2,Re10k-1,planA_colmap_108,3dgs,33.503464,0.969359,0.051509
3,Re10k-1,planA_colmap_108,scaffold,33.799107,0.971154,0.049137
4,Re10k-1,planB_vggt_108,3dgs,26.597286,0.911080,0.105110
5,Re10k-1,planB_vggt_108,scaffold,27.422918,0.921200,0.087430
6,DL3DV-2,planA_colmap_full,3dgs,34.760578,0.965748,0.067786
7,DL3DV-2,planA_colmap_full,scaffold,36.181946,0.971939,0.056363
8,DL3DV-2,planA_colmap_108,3dgs,32.287884,0.950684,0.084455
9,DL3DV-2,planA_colmap_108,scaffold,34.499348,0.965335,0.064804


/tmp/ipykernel_1381844/249872101.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  conv_df.groupby(['scene', 'setting', 'model', 'split'])['iter']


,scene,setting,model,split,min,max,count
0,Re10k-1,planA_colmap_full,3dgs,test,2000,40000,20
1,Re10k-1,planA_colmap_full,3dgs,train,2000,40000,20
2,Re10k-1,planA_colmap_full,scaffold,test,2000,40000,20
3,Re10k-1,planA_colmap_full,scaffold,train,2000,40000,20
4,Re10k-1,planA_colmap_108,3dgs,test,2000,40000,20
5,Re10k-1,planA_colmap_108,3dgs,train,2000,40000,20
6,Re10k-1,planA_colmap_108,scaffold,test,2000,40000,20
7,Re10k-1,planA_colmap_108,scaffold,train,2000,40000,20
8,Re10k-1,planB_vggt_108,3dgs,test,2000,40000,20
9,Re10k-1,planB_vggt_108,3dgs,train,2000,40000,20


## 3. Export summary tables

In [4]:
summary_cols = [
    'scene', 'plan', 'variant', 'setting', 'model', 'run_name',
    'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS'
]
summary_df = (
    final_df[summary_cols]
    .sort_values(['scene', 'setting', 'model'])
    .reset_index(drop=True)
)
summary_out = TABLE_DIR / 'part1_final_metrics_summary.csv'
summary_df.to_csv(summary_out, index=False)

paper_table = (
    final_df.pivot_table(
        index='scene',
        columns=['model', 'setting'],
        values=['ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS'],
        aggfunc='first'
    )
    .sort_index(axis=1)
)
paper_table = flatten_columns(paper_table.reset_index())
paper_table_out = TABLE_DIR / 'part1_final_metrics_wide_table.csv'
paper_table.to_csv(paper_table_out, index=False)

agg_df = (
    final_df.groupby(['model', 'setting'])[
        ['ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']
    ]
    .mean()
    .reset_index()
    .sort_values(['model', 'setting'])
    .reset_index(drop=True)
)

agg_out = TABLE_DIR / 'part1_aggregate_mean_metrics.csv'
agg_df.to_csv(agg_out, index=False)

print('saved', summary_out)
print('saved', paper_table_out)
print('saved', agg_out)

display(summary_df)
display(paper_table)
display(agg_df)

saved /home/bzhang512/CV_Project/plots/part1/tables/part1_final_metrics_summary.csv
saved /home/bzhang512/CV_Project/plots/part1/tables/part1_final_metrics_wide_table.csv
saved /home/bzhang512/CV_Project/plots/part1/tables/part1_aggregate_mean_metrics.csv


/tmp/ipykernel_1381844/70505763.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  final_df.pivot_table(
/tmp/ipykernel_1381844/70505763.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_df.groupby(['model', 'setting'])[


,scene,plan,variant,setting,model,run_name,ours_40000.PSNR,ours_40000.SSIM,ours_40000.LPIPS
0,Re10k-1,planA,colmap_full,planA_colmap_full,3dgs,default_40k_eval2k,35.164932,0.977586,0.042414
1,Re10k-1,planA,colmap_full,planA_colmap_full,scaffold,final_40k_eval2k,34.862053,0.976933,0.043847
2,Re10k-1,planA,colmap_108,planA_colmap_108,3dgs,default_40k_eval2k,33.503464,0.969359,0.051509
3,Re10k-1,planA,colmap_108,planA_colmap_108,scaffold,final_40k_eval2k,33.799107,0.971154,0.049137
4,Re10k-1,planB,vggt_108,planB_vggt_108,3dgs,default_40k_eval2k,26.597286,0.911080,0.105110
5,Re10k-1,planB,vggt_108,planB_vggt_108,scaffold,final_40k_eval2k,27.422918,0.921200,0.087430
6,DL3DV-2,planA,colmap_full,planA_colmap_full,3dgs,default_40k_eval2k,34.760578,0.965748,0.067786
7,DL3DV-2,planA,colmap_full,planA_colmap_full,scaffold,final_40k_eval2k,36.181946,0.971939,0.056363
8,DL3DV-2,planA,colmap_108,planA_colmap_108,3dgs,default_40k_eval2k,32.287884,0.950684,0.084455
9,DL3DV-2,planA,colmap_108,planA_colmap_108,scaffold,final_40k_eval2k,34.499348,0.965335,0.064804


,scene,ours_40000.LPIPS__3dgs__planA_colmap_full,ours_40000.LPIPS__3dgs__planA_colmap_108,ours_40000.LPIPS__3dgs__planB_vggt_108,ours_40000.LPIPS__scaffold__planA_colmap_full,ours_40000.LPIPS__scaffold__planA_colmap_108,ours_40000.LPIPS__scaffold__planB_vggt_108,ours_40000.PSNR__3dgs__planA_colmap_full,ours_40000.PSNR__3dgs__planA_colmap_108,ours_40000.PSNR__3dgs__planB_vggt_108,ours_40000.PSNR__scaffold__planA_colmap_full,ours_40000.PSNR__scaffold__planA_colmap_108,ours_40000.PSNR__scaffold__planB_vggt_108,ours_40000.SSIM__3dgs__planA_colmap_full,ours_40000.SSIM__3dgs__planA_colmap_108,ours_40000.SSIM__3dgs__planB_vggt_108,ours_40000.SSIM__scaffold__planA_colmap_full,ours_40000.SSIM__scaffold__planA_colmap_108,ours_40000.SSIM__scaffold__planB_vggt_108
0,Re10k-1,0.042414,0.051509,0.105110,0.043847,0.049137,0.087430,35.164932,33.503464,26.597286,34.862053,33.799107,27.422918,0.977586,0.969359,0.911080,0.976933,0.971154,0.921200
1,DL3DV-2,0.067786,0.084455,0.123195,0.056363,0.064804,0.104640,34.760578,32.287884,28.488409,36.181946,34.499348,29.638048,0.965748,0.950684,0.901901,0.971939,0.965335,0.911059
2,405841,0.125829,0.140309,0.201963,0.135362,0.148838,0.187458,32.770302,31.350622,27.503836,32.061390,30.863672,27.949621,0.937703,0.921299,0.849524,0.933910,0.918527,0.857214


,model,setting,ours_40000.PSNR,ours_40000.SSIM,ours_40000.LPIPS
0,3dgs,planA_colmap_full,34.231937,0.960346,0.078676
1,3dgs,planA_colmap_108,32.380657,0.947114,0.092091
2,3dgs,planB_vggt_108,27.529844,0.887502,0.143422
3,scaffold,planA_colmap_full,34.368463,0.960927,0.078524
4,scaffold,planA_colmap_108,33.054042,0.951672,0.087593
5,scaffold,planB_vggt_108,28.336863,0.896491,0.126510


## 4. Main figure: final metrics as paper-style dot plots

Why this replaces the old grouped bars:
- easier comparison of close metric values
- less clutter than putting 18 bars into each panel
- directly usable in the main paper text

In [5]:
dot_df = final_df[
    ['scene', 'setting', 'model', 'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']
].copy()

# ---------- normalize text ----------
dot_df['scene'] = dot_df['scene'].astype(str).str.strip()
dot_df['setting'] = dot_df['setting'].astype(str).str.strip().str.lower()
dot_df['model'] = dot_df['model'].astype(str).str.strip().str.lower()

# 如果前面解析出来的是旧名字，这里统一映射到当前画图使用的名字
setting_alias = {
    'colmap_full': 'planA_colmap_full',
    'colmap_108': 'planA_colmap_108',
    'vggt_108': 'planB_vggt_108',
    'plana_colmap_full': 'planA_colmap_full',
    'plana_colmap_108': 'planA_colmap_108',
    'planb_vggt_108': 'planB_vggt_108',
}
dot_df['setting'] = dot_df['setting'].replace(setting_alias)

combo_offsets = {
    ('planA_colmap_full', '3dgs'): -0.30,
    ('planA_colmap_full', 'scaffold'): -0.18,
    ('planA_colmap_108', '3dgs'): -0.06,
    ('planA_colmap_108', 'scaffold'): 0.06,
    ('planB_vggt_108', '3dgs'): 0.18,
    ('planB_vggt_108', 'scaffold'): 0.30,
}

# 只保留合法组合，避免 ('nan', '3dgs') 这种 key error
valid_settings = {k[0] for k in combo_offsets.keys()}
valid_models = {k[1] for k in combo_offsets.keys()}

dot_df = dot_df[dot_df['scene'].isin(SCENE_ORDER)].copy()
dot_df = dot_df[dot_df['setting'].isin(valid_settings)].copy()
dot_df = dot_df[dot_df['model'].isin(valid_models)].copy()

scene_centers = {scene: (len(SCENE_ORDER) - 1 - i) for i, scene in enumerate(SCENE_ORDER)}

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), sharey=True)

for ax, (metric, metric_label, higher_is_better) in zip(axes, METRICS):
    sub = dot_df[['scene', 'setting', 'model', metric]].copy()
    sub = sub.dropna(subset=[metric]).copy()
    sub['_is_best'] = metric_best_mask(
        sub[['scene', 'setting', 'model', metric]],
        metric,
        higher_is_better
    )

    for scene in SCENE_ORDER:
        ax.axhline(scene_centers[scene] + 0.5, color='0.90', lw=1.0, zorder=0)

    for _, row in sub.sort_values(['scene', 'setting', 'model']).iterrows():
        scene = row['scene']
        setting = row['setting']
        model = row['model']
        key = (setting, model)

        if key not in combo_offsets:
            continue

        x = row[metric]
        y = scene_centers[scene] + combo_offsets[key]

        ax.scatter(
            x, y,
            s=85,
            color=SETTING_COLORS[setting],
            marker=MODEL_MARKERS[model],
            edgecolors='black' if row['_is_best'] else 'white',
            linewidths=1.2 if row['_is_best'] else 0.8,
            zorder=3,
        )

    ax.set_title(metric_label)
    ax.grid(True, axis='x', linestyle='--', linewidth=0.6, alpha=0.35)
    ax.set_axisbelow(True)
    ax.set_yticks([scene_centers[s] for s in SCENE_ORDER])
    ax.set_yticklabels([SCENE_LABELS[s] for s in SCENE_ORDER])
    ax.set_ylim(-0.7, len(SCENE_ORDER) - 0.3)
    ax.set_xlabel(metric_label)

color_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='none',
        markerfacecolor=SETTING_COLORS[s],
        markeredgecolor='white',
        markersize=9,
        label=SETTING_LABELS[s]
    )
    for s in SETTING_ORDER if s in SETTING_COLORS and s in SETTING_LABELS
]

model_handles = [
    Line2D(
        [0], [0],
        marker=MODEL_MARKERS[m],
        color='black',
        linestyle='none',
        markersize=8,
        label=MODEL_LABELS[m]
    )
    for m in MODEL_ORDER if m in MODEL_MARKERS and m in MODEL_LABELS
]

best_handle = Line2D(
    [0], [0],
    marker='o',
    color='none',
    markerfacecolor='lightgray',
    markeredgecolor='black',
    markersize=8,
    label='Best within scene'
)

fig.legend(
    handles=color_handles + model_handles + [best_handle],
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=3
)
fig.suptitle('Part 1 Final Test Metrics (Dot Plot)', y=1.10, fontsize=14)
fig.tight_layout()

out = MAIN_DIR / 'part1_final_test_metrics_dotplot.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
plt.close(fig)
print('saved', out)

# debug: 看看实际保留下来的类别
print('settings used:', sorted(dot_df['setting'].unique()))
print('models used:', sorted(dot_df['model'].unique()))
print('scenes used:', sorted(dot_df['scene'].unique()))

saved /home/bzhang512/CV_Project/plots/part1/main/part1_final_test_metrics_dotplot.png
settings used: ['planA_colmap_108', 'planA_colmap_full', 'planB_vggt_108']
models used: ['3dgs', 'scaffold']
scenes used: ['405841', 'DL3DV-2', 'Re10k-1']


In [13]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# =========================================================
# Part 1 final metrics: compact paired dumbbell plot
# - left side shows dataset groups explicitly
# - each row = one derived setting within a dataset
# - two markers per row = 3DGS vs Scaffold-GS
# =========================================================

plot_df = final_df[
    ['scene', 'setting', 'model', 'ours_40000.PSNR', 'ours_40000.SSIM', 'ours_40000.LPIPS']
].copy()

# ---------- normalize text ----------
plot_df['scene'] = plot_df['scene'].astype(str).str.strip()
plot_df['setting'] = plot_df['setting'].astype(str).str.strip().str.lower()
plot_df['model'] = plot_df['model'].astype(str).str.strip().str.lower()

# ---------- alias mapping ----------
setting_alias = {
    'plana_colmap_full': 'planA_colmap_full',
    'plana_colmap_108': 'planA_colmap_108',
    'plana_colmap_96': 'planA_colmap_108',
    'planb_vggt_108': 'planB_vggt_108',
    'planb_vggt_96': 'planB_vggt_108',
    'colmap_full': 'planA_colmap_full',
    'colmap_108': 'planA_colmap_108',
    'colmap_96': 'planA_colmap_108',
    'vggt_108': 'planB_vggt_108',
    'vggt_96': 'planB_vggt_108',
}
plot_df['setting'] = plot_df['setting'].replace(setting_alias)

model_alias = {
    '3dgs': '3dgs',
    'gaussian-splatting': '3dgs',
    'scaffold': 'scaffold',
    'scaffold-gs': 'scaffold',
    'scaffoldgs': 'scaffold',
}
plot_df['model'] = plot_df['model'].replace(model_alias)

# ---------- local plotting config ----------
SETTING_ORDER_LOCAL = ['planA_colmap_full', 'planA_colmap_108', 'planB_vggt_108']
SETTING_LABELS_LOCAL = {
    'planA_colmap_full': 'COLMAP-full',
    'planA_colmap_108': 'COLMAP-108',
    'planB_vggt_108': 'VGGT-108',
}
SETTING_COLORS_LOCAL = {
    'planA_colmap_full': '#1f4e79',   # dark blue
    'planA_colmap_108': '#2e8b57',    # green
    'planB_vggt_108': '#9b1b45',      # wine red
}

MODEL_ORDER_LOCAL = ['3dgs', 'scaffold']
MODEL_LABELS_LOCAL = {
    '3dgs': '3DGS',
    'scaffold': 'Scaffold-GS',
}
MODEL_MARKERS_LOCAL = {
    '3dgs': 'o',
    'scaffold': 's',
}

# ---------- keep only valid rows ----------
valid_settings = set(SETTING_ORDER_LOCAL)
valid_models = set(MODEL_ORDER_LOCAL)

plot_df = plot_df[plot_df['scene'].isin(SCENE_ORDER)].copy()
plot_df = plot_df[plot_df['setting'].isin(valid_settings)].copy()
plot_df = plot_df[plot_df['model'].isin(valid_models)].copy()

# ---------- build row order ----------
row_defs = []
for scene in SCENE_ORDER:
    for setting in SETTING_ORDER_LOCAL:
        sub = plot_df[(plot_df['scene'] == scene) & (plot_df['setting'] == setting)]
        if len(sub) > 0:
            row_defs.append({'scene': scene, 'setting': setting})

for i, rd in enumerate(row_defs):
    rd['y'] = i

yticklabels = [SETTING_LABELS_LOCAL[rd['setting']] for rd in row_defs]

scene_to_rows = {}
for scene in SCENE_ORDER:
    rows = [rd['y'] for rd in row_defs if rd['scene'] == scene]
    if rows:
        scene_to_rows[scene] = rows

# ---------- figure ----------
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharey=True)

for ax, (metric, metric_label, higher_is_better) in zip(axes, METRICS):
    # subtle scene group background
    for j, scene in enumerate(SCENE_ORDER):
        rows = scene_to_rows.get(scene, [])
        if not rows:
            continue
        ymin = min(rows) - 0.5
        ymax = max(rows) + 0.5
        ax.axhspan(
            ymin, ymax,
            color='0.985' if j % 2 == 0 else '0.965',
            zorder=0
        )

    # separators between datasets
    for scene in SCENE_ORDER[:-1]:
        rows = scene_to_rows.get(scene, [])
        if rows:
            ax.axhline(max(rows) + 0.5, color='0.72', lw=1.0, zorder=1)

    # plot each row
    for rd in row_defs:
        scene = rd['scene']
        setting = rd['setting']
        y = rd['y']

        sub = plot_df[
            (plot_df['scene'] == scene) &
            (plot_df['setting'] == setting)
        ][['model', metric]].dropna()

        if len(sub) == 0:
            continue

        vals = {}
        for model in MODEL_ORDER_LOCAL:
            model_sub = sub[sub['model'] == model][metric]
            if len(model_sub) > 0:
                vals[model] = float(model_sub.iloc[0])

        # connecting line
        if len(vals) == 2:
            ax.plot(
                [vals['3dgs'], vals['scaffold']],
                [y, y],
                color='0.72',
                lw=1.4,
                zorder=2
            )

        # best model within the row
        if len(vals) > 0:
            best_model = max(vals, key=vals.get) if higher_is_better else min(vals, key=vals.get)

            for model in MODEL_ORDER_LOCAL:
                if model not in vals:
                    continue

                x = vals[model]

                # base marker
                ax.scatter(
                    x, y,
                    s=80,
                    marker=MODEL_MARKERS_LOCAL[model],
                    color=SETTING_COLORS_LOCAL[setting],
                    edgecolors='white',
                    linewidths=0.8,
                    zorder=3
                )

                # transparent highlight ring for the better one
                if model == best_model:
                    ax.scatter(
                        x, y,
                        s=155,
                        marker='o',
                        facecolors='none',
                        edgecolors='black',
                        linewidths=1,
                        zorder=4
                    )

    ax.set_title(metric_label, fontsize=16, fontweight='bold')
    ax.grid(True, axis='x', linestyle='--', linewidth=0.55, alpha=0.30)
    ax.set_axisbelow(True)
    ax.set_xlabel(metric_label, fontsize=13)

    ax.set_yticks(range(len(row_defs)))
    if ax is axes[0]:
        ax.set_yticklabels(yticklabels, fontsize=11)
        ax.tick_params(axis='y', length=0, pad=6)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis='y', length=0)

    ax.invert_yaxis()

# ---------- add dataset labels on far left of first axis ----------
ax0 = axes[0]
for scene in SCENE_ORDER:
    rows = scene_to_rows.get(scene, [])
    if not rows:
        continue
    y_center = 0.5 * (min(rows) + max(rows))
    ax0.text(
        -0.17, y_center,
        SCENE_LABELS.get(scene, scene),
        transform=ax0.get_yaxis_transform(),
        ha='right',
        va='center',
        fontsize=12.5,
        fontweight='bold'
    )

# small header for the two left-side levels
# ax0.text(
#     -0.17, -0.95,
#     'Dataset',
#     transform=ax0.get_yaxis_transform(),
#     ha='right',
#     va='bottom',
#     fontsize=12,
#     fontweight='bold'
# )
# ax0.text(
#     -0.005, -0.95,
#     'Setting',
#     transform=ax0.get_yaxis_transform(),
#     ha='right',
#     va='bottom',
#     fontsize=12,
#     fontweight='bold'
# )

# ---------- legends ----------
setting_handles = [
    Patch(facecolor=SETTING_COLORS_LOCAL[s], edgecolor='none', label=SETTING_LABELS_LOCAL[s])
    for s in SETTING_ORDER_LOCAL
]

model_handles = [
    Line2D(
        [0], [0],
        marker=MODEL_MARKERS_LOCAL[m],
        color='black',
        linestyle='none',
        markerfacecolor='black',
        markersize=8,
        label=MODEL_LABELS_LOCAL[m]
    )
    for m in MODEL_ORDER_LOCAL
]

best_handle = Line2D(
    [0], [0],
    marker='o',
    color='black',
    linestyle='none',
    markerfacecolor='none',
    markersize=10,
    markeredgewidth=1.6,
    label='Better of the two'
)

# first legend: settings
fig.legend(
    handles=setting_handles,
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(0.33, 1.04),
    ncol=3,
    fontsize=12,
    title='Derived setting',
    title_fontsize=12.5
)

# second legend: model + better ring
fig.legend(
    handles=model_handles + [best_handle],
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(0.76, 1.04),
    ncol=3,
    fontsize=12,
    title='Model / highlight',
    title_fontsize=12.5
)

fig.suptitle('Part 1 Final Test Metrics (Paired Comparison)', y=1.11, fontsize=17)
fig.tight_layout(rect=[0.08, 0.02, 1.00, 0.92])

out = MAIN_DIR / 'part1_final_test_metrics_paired_compact.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
plt.close(fig)

print('saved', out)
print('settings used:', sorted(plot_df['setting'].unique()))
print('models used:', sorted(plot_df['model'].unique()))
print('scenes used:', sorted(plot_df['scene'].unique()))

saved /home/bzhang512/CV_Project/plots/part1/main/part1_final_test_metrics_paired_compact.png
settings used: ['planA_colmap_108', 'planA_colmap_full', 'planB_vggt_108']
models used: ['3dgs', 'scaffold']
scenes used: ['405841', 'DL3DV-2', 'Re10k-1']


## 5. Main figure: test PSNR convergence arranged as scene × model

This layout is better than a scene × setting grid because the comparison of initialization settings now happens **inside each panel**, which matches the paper narrative.

In [7]:
fig, axes = plt.subplots(len(SCENE_ORDER), len(MODEL_ORDER), figsize=(12.5, 10), sharex=True)

for r, scene in enumerate(SCENE_ORDER):
    for c, model in enumerate(MODEL_ORDER):
        ax = axes[r, c]
        sub = conv_df[
            (conv_df['scene'] == scene) &
            (conv_df['model'] == model) &
            (conv_df['split'] == 'test')
        ].copy()

        for setting in SETTING_ORDER:
            cur = sub[sub['setting'] == setting].sort_values('iter')
            if cur.empty:
                continue
            ax.plot(
                cur['iter'], cur['psnr'],
                color=SETTING_COLORS[setting],
                linestyle='-',
                linewidth=2.2,
                label=SETTING_LABELS[setting],
            )

        if r == 0:
            ax.set_title(MODEL_LABELS[model])
        if c == 0:
            ax.set_ylabel(f'{SCENE_LABELS[scene]}\nPSNR ↑')

        ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
        ax.set_axisbelow(True)

for ax in axes[-1, :]:
    ax.set_xlabel('Iteration')

setting_handles = [
    Line2D([0], [0], color=SETTING_COLORS[s], lw=2.5, label=SETTING_LABELS[s])
    for s in SETTING_ORDER
]
fig.legend(handles=setting_handles, frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Part 1 Test PSNR Convergence', y=1.05, fontsize=14)
fig.tight_layout()

out = MAIN_DIR / 'part1_test_psnr_convergence_scene_by_model.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
plt.close(fig)
print('saved', out)
plt.show()

saved /home/bzhang512/CV_Project/plots/part1/main/part1_test_psnr_convergence_scene_by_model.png


## 6. Appendix figure: test L1 convergence (same scene × model layout)

In [8]:
fig, axes = plt.subplots(len(SCENE_ORDER), len(MODEL_ORDER), figsize=(12.5, 10), sharex=True)

for r, scene in enumerate(SCENE_ORDER):
    for c, model in enumerate(MODEL_ORDER):
        ax = axes[r, c]
        sub = conv_df[
            (conv_df['scene'] == scene) &
            (conv_df['model'] == model) &
            (conv_df['split'] == 'test')
        ].copy()

        for setting in SETTING_ORDER:
            cur = sub[sub['setting'] == setting].sort_values('iter')
            if cur.empty:
                continue
            ax.plot(
                cur['iter'], cur['l1'],
                color=SETTING_COLORS[setting],
                linestyle='-',
                linewidth=2.2,
                label=SETTING_LABELS[setting],
            )

        if r == 0:
            ax.set_title(MODEL_LABELS[model])
        if c == 0:
            ax.set_ylabel(f'{SCENE_LABELS[scene]}\nL1 ↓')

        ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
        ax.set_axisbelow(True)

for ax in axes[-1, :]:
    ax.set_xlabel('Iteration')

setting_handles = [
    Line2D([0], [0], color=SETTING_COLORS[s], lw=2.5, label=SETTING_LABELS[s])
    for s in SETTING_ORDER
]
fig.legend(handles=setting_handles, frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Appendix: Test L1 Convergence', y=1.05, fontsize=14)
fig.tight_layout()

out = APP_DIR / 'part1_test_l1_convergence_scene_by_model.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
plt.close(fig)
print('saved', out)

saved /home/bzhang512/CV_Project/plots/part1/appendix/part1_test_l1_convergence_scene_by_model.png


## 7. Appendix figure: train/test overlay diagnostics

This remains a diagnostic figure, not a main paper figure.  
By default it uses the first scene in `SCENE_ORDER`, but you can switch it if another scene looks more illustrative.

In [9]:
SCENE_FOR_OVERLAY = SCENE_ORDER[0]

fig, axes = plt.subplots(2, len(SETTING_ORDER), figsize=(16, 8), sharex=True)

for c, setting in enumerate(SETTING_ORDER):
    ax_psnr = axes[0, c]
    ax_l1 = axes[1, c]
    sub = conv_df[
        (conv_df['scene'] == SCENE_FOR_OVERLAY) &
        (conv_df['setting'] == setting)
    ].copy()

    for model in MODEL_ORDER:
        for split, alpha, lw in [('test', 1.0, 2.2), ('train', 0.50, 1.7)]:
            cur = sub[(sub['model'] == model) & (sub['split'] == split)].sort_values('iter')
            if cur.empty:
                continue

            ax_psnr.plot(
                cur['iter'], cur['psnr'],
                color='black',
                linestyle=MODEL_LINESTYLES[model],
                linewidth=lw,
                alpha=alpha,
                marker=MODEL_MARKERS[model] if split == 'test' else None,
                markersize=3.5,
            )
            ax_l1.plot(
                cur['iter'], cur['l1'],
                color='black',
                linestyle=MODEL_LINESTYLES[model],
                linewidth=lw,
                alpha=alpha,
                marker=MODEL_MARKERS[model] if split == 'test' else None,
                markersize=3.5,
            )

    ax_psnr.set_title(SETTING_LABELS[setting])
    ax_psnr.set_ylabel('PSNR ↑')
    ax_l1.set_ylabel('L1 ↓')
    ax_l1.set_xlabel('Iteration')

    ax_psnr.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
    ax_l1.grid(True, linestyle='--', linewidth=0.6, alpha=0.28)
    ax_psnr.set_axisbelow(True)
    ax_l1.set_axisbelow(True)

method_handles = [
    Line2D([0], [0], color='black', linestyle=MODEL_LINESTYLES[m], marker=MODEL_MARKERS[m], lw=2.0, label=MODEL_LABELS[m])
    for m in MODEL_ORDER
]
split_handles = [
    Line2D([0], [0], color='black', lw=2.0, alpha=1.0, label='Test'),
    Line2D([0], [0], color='black', lw=2.0, alpha=0.50, label='Train'),
]
leg1 = axes[0, 0].legend(handles=method_handles, frameon=False, title='Method', loc='best')
axes[0, 0].add_artist(leg1)
axes[1, -1].legend(handles=split_handles, frameon=False, title='Split', loc='best')

fig.suptitle(f'Appendix: Train/Test Overlay Diagnostics ({SCENE_FOR_OVERLAY})', y=1.03, fontsize=14)
fig.tight_layout()

out = APP_DIR / f'part1_train_test_overlay_{SCENE_FOR_OVERLAY.lower().replace("-", "").replace(" ", "_")}.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
plt.close(fig)
print('saved', out)

saved /home/bzhang512/CV_Project/plots/part1/appendix/part1_train_test_overlay_re10k1.png


## 8. Appendix figure: aggregate mean heatmaps

These are not a replacement for the main quantitative table.  
They are useful as a quick visual summary when discussing global trends across all three scenes.

In [10]:
agg_plot_df = agg_df.copy()

heatmaps = []
for metric, metric_label, higher_is_better in METRICS:
    heat = (
        agg_plot_df.pivot(index='model', columns='setting', values=metric)
        .reindex(index=MODEL_ORDER, columns=SETTING_ORDER)
    )
    heatmaps.append((metric, metric_label, higher_is_better, heat))

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6))

for ax, (metric, metric_label, higher_is_better, heat) in zip(axes, heatmaps):
    arr = heat.values.astype(float)

    cmap = 'viridis' if higher_is_better else 'magma_r'
    im = ax.imshow(arr, aspect='auto', cmap=cmap)

    ax.set_title(metric_label)
    ax.set_xticks(np.arange(len(SETTING_ORDER)))
    ax.set_xticklabels([SETTING_LABELS[s] for s in SETTING_ORDER], rotation=20, ha='right')
    ax.set_yticks(np.arange(len(MODEL_ORDER)))
    ax.set_yticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            ax.text(j, i, f'{val:.3f}' if metric != 'ours_40000.PSNR' else f'{val:.2f}',
                    ha='center', va='center', color='white', fontsize=10, weight='semibold')

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=9)

fig.suptitle('Appendix: Aggregate Mean Metrics Across Scenes', y=1.04, fontsize=14)
fig.tight_layout()

out = APP_DIR / 'part1_aggregate_mean_metrics_heatmaps.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
plt.close(fig)
print('saved', out)

saved /home/bzhang512/CV_Project/plots/part1/appendix/part1_aggregate_mean_metrics_heatmaps.png


## 9. Qualitative figure placeholder

Quantitative plots are generated here.  
For the report, the qualitative rendering comparison should usually be assembled from your rendered images rather than from CSV files:

Recommended qualitative layout for the paper:
- choose **2 scenes**
- choose **2–3 novel views per scene**
- columns: **GT / COLMAP full / COLMAP-108 / VGGT-108**
- optionally split by **3DGS vs Scaffold-GS** if space permits

Suggested scene priority:
1. **DL3DV-2** as the most discriminative scene
2. **Re10k-1** as the clean high-quality showcase
3. **405841** as an appendix harder case

## 10. Show generated files

In [11]:
for p in sorted(MAIN_DIR.glob('*')):
    print('MAIN    ', p.name)
for p in sorted(APP_DIR.glob('*')):
    print('APPENDIX', p.name)
for p in sorted(TABLE_DIR.glob('*')):
    print('TABLE   ', p.name)

MAIN     part1_final_test_metrics_dotplot.png
MAIN     part1_final_test_metrics_dumbbell.png
MAIN     part1_final_test_metrics_paired_compact.png
MAIN     part1_test_psnr_convergence_scene_by_model.png
APPENDIX part1_aggregate_mean_metrics_heatmaps.png
APPENDIX part1_test_l1_convergence_scene_by_model.png
APPENDIX part1_train_test_overlay_re10k1.png
TABLE    part1_aggregate_mean_metrics.csv
TABLE    part1_final_metrics_summary.csv
TABLE    part1_final_metrics_wide_table.csv
